# Lesson 1.1: Cleaning the messy penguin file

The messy copy of the penguin data has problems added on purpose: duplicate rows, labels written in different ways, zeros used for "not measured" and two typos. Here we find each one and clean it, step by step, following the routine from the lesson. Run each cell from top to bottom with **Shift + Enter**.

In [1]:
from pathlib import Path
import pandas as pd

# Find the data files, wherever this notebook is opened from
here = Path.cwd()
folder = next(p / "data/penguins" for p in [here, *here.parents] if (p / "data/penguins/penguins-messy.csv").exists())

# Step 1: keep the original untouched, and clean a copy
original = pd.read_csv(folder / "penguins-messy.csv")
df = original.copy()

# Step 2: count the rows before we start
print("rows before:", len(df))

rows before: 348


## Step 3: look for gaps, then duplicates

`isna()` finds empty cells. Zeros are not empty, so they don't show up here. We deal with them below.

In [2]:
print(df.isna().sum())
print()
print("exact duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)]

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

exact duplicate rows: 4


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
15,Adelie,Torgersen,36.6,17.8,185.0,3700.0,FEMALE
95,Adelie,Dream,40.8,18.9,208.0,4300.0,MALE
180,Chinstrap,Dream,46.4,17.8,191.0,3700.0,FEMALE
280,Gentoo,Biscoe,45.3,13.8,208.0,4200.0,FEMALE
344,Adelie,Torgersen,36.6,17.8,185.0,3700.0,FEMALE
345,Adelie,Dream,40.8,18.9,208.0,4300.0,MALE
346,Chinstrap,Dream,46.4,17.8,191.0,3700.0,FEMALE
347,Gentoo,Biscoe,45.3,13.8,208.0,4200.0,FEMALE


In [3]:
# Exact duplicates: keep the first copy, remove the rest
df = df.drop_duplicates()
print("rows now:", len(df))

rows now: 344


## Step 4: sort each number column and check both ends

In [4]:
for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    values = df[col].dropna().sort_values()
    print(f"{col:18} lowest {values.head(3).tolist()}   highest {values.tail(3).tolist()}")

bill_length_mm     lowest [32.1, 33.1, 33.5]   highest [55.9, 58.0, 59.6]
bill_depth_mm      lowest [13.1, 13.2, 13.3]   highest [21.2, 21.2, 21.5]
flipper_length_mm  lowest [172.0, 174.0, 176.0]   highest [230.0, 231.0, 1980.0]
body_mass_g        lowest [0.0, 0.0, 2700.0]   highest [6050.0, 6300.0, 42500.0]


Two penguins weigh 0 g, one weighs 42,500 g and one flipper is 1,980 mm long. The outlier rule from the lesson flags them too: take the middle half of the data (between the 25th and 75th percentiles) and flag anything more than 1.5 times that range beyond it.

In [5]:
# The rule on the messy file as it came, duplicates included, like in the lesson
q1, q3 = original["body_mass_g"].quantile([0.25, 0.75])
low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print(f"middle half: {q1:.0f} to {q3:.0f} g, flag below {low:.0f} or above {high:.0f} g")
mass = df["body_mass_g"]
df[(mass < low) | (mass > high)]

middle half: 3550 to 4750 g, flag below 1750 or above 6550 g


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
60,Adelie,Biscoe,35.7,16.9,185.0,0.0,FEMALE
75,Adelie,Torgersen,42.8,18.5,195.0,42500.0,MALE
200,Chinstrap,Dream,51.5,18.7,187.0,0.0,MALE


In [6]:
# The zeros are missing values in disguise: turn them back into empty cells
df.loc[df["body_mass_g"] == 0, "body_mass_g"] = None

# 42,500 g is 4,250 with an extra zero, and 1,980 mm is 198 with one. We can confirm both
# against the real file, so we fix them. Without a way to confirm, we would remove them instead.
df.loc[df["body_mass_g"] == 42500, "body_mass_g"] = 4250
df.loc[df["flipper_length_mm"] == 1980, "flipper_length_mm"] = 198
print("heaviest penguin now:", df["body_mass_g"].max(), "g")

heaviest penguin now: 6300.0 g


## Step 5: count the labels in each text column

In [7]:
df["species"].map(repr).value_counts()

species
'Adelie'            150
'Gentoo'            123
'Chinstrap'          68
'adelie'              1
'Adelie '             1
'Gentoo penguin'      1
Name: count, dtype: int64

In [8]:
# Remove extra spaces, use one spelling, and map the variations to the right name
df["species"] = df["species"].str.strip().str.capitalize().replace({"Gentoo penguin": "Gentoo"})
df["species"].value_counts()

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

## Filling the gaps

Typical weights differ a lot by species and sex, so the overall median would be wrong for most groups.

In [9]:
print("overall median:", df["body_mass_g"].median(), "g")
df.groupby(["species", "sex"])["body_mass_g"].median()

overall median: 4050.0 g


species    sex   
Adelie     FEMALE    3400.0
           MALE      4000.0
Chinstrap  FEMALE    3550.0
           MALE      3950.0
Gentoo     FEMALE    4700.0
           MALE      5500.0
Name: body_mass_g, dtype: float64

In [10]:
# The two penguins with no measurements at all hold almost nothing, so we drop them
df = df.dropna(subset=["bill_length_mm", "flipper_length_mm"], how="all")

# The two zeros become the median for their species and sex
group_median = df.groupby(["species", "sex"])["body_mass_g"].transform("median")
df["body_mass_g"] = df["body_mass_g"].fillna(group_median)
print("rows now:", len(df))

rows now: 342


## Before and after

Here we compare with the real file. Notice that the errors moved the average, but not the median.

In [11]:
real = pd.read_csv(folder / "penguins.csv")
summary = pd.DataFrame({
    "messy copy": [len(original), original["species"].nunique(), original["body_mass_g"].max(), original["body_mass_g"].mean(), original["body_mass_g"].median()],
    "real file": [len(real), real["species"].nunique(), real["body_mass_g"].max(), real["body_mass_g"].mean(), real["body_mass_g"].median()],
}, index=["rows", "species labels", "heaviest (g)", "average (g)", "median (g)"])
summary.round(1)

,messy copy,real file
rows,348.0,344.0
species labels,6.0,3.0
heaviest (g),42500.0,6300.0
average (g),4291.2,4201.8
median (g),4050.0,4050.0


Our cleaned copy has 342 rows, because we also dropped the two penguins with no measurements, and filled the two zeros with group medians. The real file keeps those two empty rows, which is why the lesson says "back to 344".

## Step 6: write down every change

| Change | Why |
|---|---|
| Removed 4 exact duplicate rows | Each penguin should count once |
| Turned 2 body masses of 0 g into missing values, then filled them with the median for their species and sex | Nobody weighed them; 0 g is impossible |
| Fixed 42,500 g to 4,250 g and 1,980 mm to 198 mm | Typos with an extra zero, confirmed against the real file |
| Mapped "adelie", "Adelie " and "Gentoo penguin" to "Adelie" and "Gentoo" | One spelling per species |
| Dropped the 2 penguins with no measurements | They hold almost nothing |